## DATA 442 / 610 - Deep Learning and Neural Networks
## Assignment 1: Building a NN from Scratch
## Mark J Serena - W&M ID # 930657215

In [ ]:
from collections import Counter
from pathlib import Path

import datetime
from datetime import datetime as dt
import math
import os
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

print(f"Python {sys.version}")
print(f"sklearn {sklearn.__version__}")

## Problem 1 (50 points) - Fashion-MNIST classification from scratch

### Part 1: Data Preprocessing (10 points)

### Question 1 (5 points): Visualize one example from each class with the appropriate label.

In [ ]:
from sklearn.datasets import fetch_openml

fashion_mnist = fetch_openml("Fashion-MNIST", version=1)
X = fashion_mnist.data.to_numpy(dtype=np.float32)
y = fashion_mnist.target.to_numpy(dtype=np.int8)

In [ ]:
# Print label numbers
label_numbers = np.unique(y)
print(f"label numbers {label_numbers}")

label_names = [
    'T-shirt/Top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

for i, name in enumerate(label_names):
    print(i, "=", name)

In [ ]:
# This code came from a previous course, DATA 209 - Linear Algebra,
# that involved using MNIST-Fashion with Multilayer Perceptron for Classification
# It displays sample images of each of the ten clothing types
# Its more complicated than it needs to be for this

# Each category will go on a row
m = len(label_names)

# Specify number of columns
# In case you want multiple images
n = 1

# Configure a subplot grid
fig, ax = plt.subplots(m, n, 
                       figsize=(5, 10), # this controls the figure size
                       gridspec_kw={'hspace': 0.1, 'wspace': 0.1},
                       squeeze=False)

fig.subplots_adjust(top=0.90)
# Loop over each of the clothing types
for yi in range(m):
    
    # Name of this type of product
    label = str(yi) + " - " + label_names[yi] 
    
    # Get all image indices of this type
    image_indices = np.where(y == yi)[0]

    # Select a random subset to display
    random_indices = np.random.choice(image_indices, n, replace=False)
    
    # Show the images    
    for a, idx in zip(ax[yi], random_indices):
        # Reshape the image into a 28x28 array
        img = X[idx, :].reshape(28, 28)

        # Show the image
        a.imshow(img, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
        
        # Turn off tick marks
        a.set_xticks([])
        a.set_yticks([])

    # Add label name and number at the end of the row
    ax[yi, -1].set_ylabel(label, rotation=0, ha='left')
    ax[yi, -1].yaxis.set_label_position('right')
fig.suptitle(
    'Question 1:\nVisualize one example from each class\nwith the appropriate label.',
    ha='left',
    fontweight='bold'
)


plt.savefig("Assignment1_P1_Q1.png")
plt.show()

### Question 2 (3 points): Perform a principal component analysis (PCA) of the input data. 

You may use sklearn.decomposition.PCA. Visualize the dataset along the first two principal component dimensions. Comment on the data distribution. How many principal components are necessary to capture 80% of dataset variation?

### What is PCA?

PCA (Principal Component Analysis) is a technique for dimensionality reduction. Reducing the number of features (independent variables) in a dataset while trying to preserve as much of its important information as possible.

The Fashion-MNIST dataset consists of 28 pixels x 28 pixel images. When each pixel is treated as a feature, 28x28=784 features (dimensions)!  How can this be reduced while still maintaining what makes the image distinct. PCA doesn't specifically know what makes an image distinct, but it can identify what accounts for the most variation in the data. These variations can capture important characteristics that distinguish the images from one another.

PCA finds combinations of the original 784 features that capture the greatest amount of variation in the images. These combinations are called principal components.


In [ ]:
# Normally when using PCA a standard scaler is used
# but with the Fashion-MNIST data each pixel uses the same 
# gray scale values

from sklearn.decomposition import PCA

# Create a PCA object that will reduce the original dataset
# to two principal components.
pca = PCA(n_components=2)

# Fit PCA to X and transform the original observations from 
# 784-D into a 2-D PCA space.
# X_pca isn't a subset of X. It's the same 70,000 observations, 
# but instead of each observation having 784 features it now has just 2.
X_pca = pca.fit_transform(X)

# Now be a bit clever
# label_names is a list of the 10 different clothing types
# in order of their associated label number, 0,1,2,3,....
# For each label name you'll have the number(i) and the name
for i, name in enumerate(label_names):
    # Create a Boolean mask that identifies all observations
    # in y that belong to the current label name.
    mask = y == i
    # Plot only the observations belonging to the current label name.
    #
    # X_pca[mask, 0] selects the PC1 value for each observation
    # belonging to the current label name and is plotted on the x-axis.
    #
    # X_pca[mask, 1] selects the PC2 value for each observation
    # belonging to the current label name and is plotted on the y-axis.
    #
    
    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        label=name
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(loc='center left', bbox_to_anchor=[1, 0.5])

# Calculate the percentage of total dataset variance
# explained by the principal components in the PCA model.
explained_variance_percent = round(pca.explained_variance_ratio_.sum() * 100)
# Every picture tells a story
plot_title = "MNIST-Fashion Data - PCA (PC1 vs PC2)" + "\n"

plt.title(plot_title)
plt.savefig("Assignment1_P1_Q2.png")
plt.show()


### Comments on the distribution

In spite of the data containing an equal number of observations (7,000) for each label, the distribution is not composed of 10 distinct clusters. By using just the first two principal components, we are visualizing only about 46.8% of the total variation in the dataset.

Some classes do form relatively distinct clusters. For example, ankle boots, shirts, and bags appear to be fairly well separated, while several of the other clothing classes have considerable overlap.

In [ ]:
# Fit PCA keeping all components (no n_components given), so we can see how
# variance is distributed across every principal component.
pca = PCA()
pca.fit(X)

# explained_variance_ratio_ is the fraction of total variance captured by each
# component, sorted from largest to smallest. Multiply by 100 for percentages.
variance_percentages = pca.explained_variance_ratio_ * 100

# Running (cumulative) total of variance explained so far.
total_pct = 0

# Walk through the components in order, starting the count at 1 so the printed
# numbers match the usual "PC1, PC2, ..." naming.
for i, pct in enumerate(variance_percentages, start=1):
    total_pct += pct

    # Still below the 80% target: report this component's individual
    # contribution and the cumulative total so far.
    if total_pct < 80:
        print(f"Principal Component {i}: {pct:.2f}%\t"
              f"Variance Total {total_pct:.2f}%")

    # Target reached: report how many components it took, then stop.
    # Since components are sorted by variance, i is the smallest number of
    # components that reaches 80%.
    if total_pct >= 80:
        print(f"\n{i} principal components are needed to capture 80% of the variance.")
        break

### How many principal components are necessary to capture 80% of dataset variation?

See list above, but 24 components make up 80% of the dataset variance. That's pretty amazing given there are 784. 

### Question 3 (2 points): Perform a train/validation/test split and scale the input features using either min-max scaling or standard scaling. 

You may use sklearn.preprocessing.

In [ ]:
from sklearn.model_selection import train_test_split

# Always split the original data before scaling 
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Split again from the temp data to create validation and test sets
# You should have 80% for training, then 10% each for validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)


In [ ]:
from sklearn.preprocessing import StandardScaler
# Fit the scaler on just the training data. This means
# learning the scaling parameters from X_train.
# With Standard Scaler, it learns the mean and standard deviation 
# of each feature in the training data.
# Its not changing the data, that occurs with transform.
scaler = StandardScaler()
scaler.fit(X_train)

# Apply that same scaler on all three sets
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
# Convert y (the label number 0 thru 9) to a one hot format
# in all the y arrays
y_train = np.eye(10, dtype=np.float32)[y_train]
y_val = np.eye(10, dtype=np.float32)[y_val]
y_test = np.eye(10, dtype=np.float32)[y_test]

## Part 2: Model training (40 points)

Implement a MLP with two hidden layers to classify the MNIST-Fashion dataset. Use the softmax
activation on the output layer.

Train using the categorical cross entropy loss.


### Question 1 (10 points): 

Modify the model initialization to include a second hidden layer of
neurons and neuron weights, and modify the “forward" function to include the softmax output.

### See the documented MLP below as well as its implemtation

### First need to build an MLP with this structure

$ X \rightarrow \text{Hidden Layer 1}
\rightarrow h_1
\rightarrow \text{Hidden Layer 2}
\rightarrow h_2
\rightarrow \text{Multiclass Output Layer}
\rightarrow \hat y$


### MLP Class and Functions

- __init__() — [Class method / automatic]
  — Initializes the model when you create MLP_3Layer(...).

  - _initialize_weights() — [Internal class method]
    — Initializes the weights and biases for Hidden Layer 1, Hidden Layer 2, and the output layer.

- train() — [External function]
  — Controls the mini-batch SGD training process.

  - batch_loader() — [External helper function]
    — Shuffles the training data and produces mini-batches.

  - forward() — [Class method]
    — Performs forward propagation through Hidden Layer 1,
      Hidden Layer 2, and the output layer.

  - backward() — [Class method]
    — Performs backpropagation through the output layer,
      Hidden Layer 2, and Hidden Layer 1 and calculates how the loss changes with each weight and bias, then returns those gradients.

  - weight_update() — [Class method]
    — Updates the weights and biases for all three trainable
      layers using gradient descent.

  - categorical_cross_entropy_loss() — [External helper function]
    — Calculates scalar value that quantifies how far off the model's 
        predictions are from the true labels.

  - accuracy() — [External helper function]
    — Calculates classification accuracy from the model's predictions.


- forward() — [Class method]
  — Can also be called separately after training to produce predictions.

  - The predictions returned by forward() can then be passed separately to categorical_cross_entropy_loss() and accuracy().

### Functions used by MLP

- preactivation(x, w, b)
    - Calculates the preactivation: z = XW + b.
    - Used during forward propagation.
- grad_preactivation(x, w, b, z)
    - Calculates the gradients associated with the preactivation calculation z = XW + b.
    - Used during backpropagation.
- softmax(z)
    - Applies the softmax activation function.
    - Used on the output layer for multiclass classification.
- relu(z)
    - Applies the ReLU activation function.
    - Used on Hidden Layer 1 and Hidden Layer 2.
- grad_relu(z, a)
    - Calculates the gradient of ReLU.
    - Used during backpropagation through the hidden layers.
- categorical_cross_entropy_loss(y_true, y_pred)
    - Calculates categorical cross-entropy loss.

In [ ]:
# Functions used by MLP

def preactivation(x, w, b):
    """ Applies weights and biases
    """
    return np.dot(x, w) + b

def grad_preactivation(x, w, b, z):
    """ Accepts inputs x, w, b, output z
        Returns gradient of output w.r.t. each input
    """
    dz_dx = w
    dz_dw = x
    dz_db = 1.
    return dz_dx, dz_dw, dz_db

def softmax(z):
    """
    Purpose: Designed for multi-class classification tasks, 
    where an input must be assigned to one of several mutually exclusive classes.
    
    Output: Converts a vector of raw scores (logits) into probabilities for each class, 
    ensuring that all probabilities sum to 1.
    
    Behavior: Softmax amplifies the largest input value, making it more likely that the 
    highest-scoring class will have a probability close to 1, while the others 
    are pushed closer to 0. 
    """
    # Subtract the largest value in each observation for numerical stability.
    shifted_z = z - np.max(z, axis=1, keepdims=True)

    # Calculate e^z for each output value.
    exp_z = np.exp(shifted_z)

    # Convert the values into probabilities.
    # The probabilities for each observation will sum to 1.
    probabilities = exp_z / np.sum(exp_z, axis=1, keepdims=True)

    return probabilities

def relu(z):
    """ Thresholded ramp activation function
    """
    return np.maximum(0, z)

def grad_relu(z, a):
    """ Accepts input z, output a
        Returns gradient of output w.r.t. input
    """
    return np.where(z >= 0, 1., 0.)

def categorical_cross_entropy_loss(y_true, y_pred, eps=1e-10):
    """
    Purpose:
        Calculates the multiclass cross-entropy loss.

    Input:
        y_true - One-hot encoded true class labels.
        y_pred - Predicted class probabilities from softmax.
        eps    - Small value added to prevent log(0).

    Output:
        loss - Average multiclass cross-entropy loss.
    """

    # Calculate the cross-entropy loss for the batch.
    loss = -np.sum(y_true * np.log(y_pred + eps)) / y_true.shape[0]

    return loss

def accuracy(y_true, y_pred):
    """
    Purpose:
        Calculates the fraction of predictions that are correct.

    Input:
        y_true - One-hot encoded true class labels.
        y_pred - Predicted class probabilities from softmax.

    Output:
        accuracy_score - Accuracy between 0 and 1.
    """

    # Find the class number where each true label has its 1.
    true_classes = np.argmax(y_true, axis=1)

    # Find the class number with the highest predicted probability.
    predicted_classes = np.argmax(y_pred, axis=1)

    # Compare the predictions with the true classes.
    correct_predictions = predicted_classes == true_classes

    # Divide the number of correct predictions by the total.
    number_correct = np.sum(correct_predictions)
    total_predictions = len(true_classes)
    accuracy_score = number_correct / total_predictions

    return accuracy_score

### Question 2 (5 points): Derive d_loss__d_z_output, the gradient of the loss with respect to the final layer pre-activation.

The code for this is incredibly simple:
        `d_loss__d_z_o = (a_o - y_true) / B`

The math behind it, not so simple (at least to me).

First, what is d_loss__d_z_o? It is the derivative of the loss with respect to the output layer’s logits, the raw scores before softmax converts them into probabilities.

The equation subtracts the one-hot encoded true labels (y_true) from the predicted probabilities (a_o), then divides by the number of observations in the batch (B). We divide by B because our loss is averaged across the batch.

- The softmax equation - softmax converts the output logits into class probabilities:

$p_j = \frac{e^{z_j}}{\sum_{k=1}^{C} e^{z_k}}$

- The categorical cross-entropy formula 

$\ell = -\sum_{j=1}^{C} y_j \times \ln(p_j)$

Where \(C\) is the number of classes or categories

There's an expression that "it all comes out in the wash", but we can paraphrase that into "it all comes out in substitution and differentiation". Replace $p_j$ in the categorical cross-entropy formula with the softmax formula. You now have

$$
\ell = -\sum_{j=1}^{C} y_j \times \ln\left(\frac{e^{z_j}}{\sum_{k=1}^{C} e^{z_k}}\right)
$$

Looks really scary, but using the log rule and the chain rule to find the derivative we go thru these steps

First, use the log rules to expand the expression, meaning rewrite the logarithm using two rules:

1) $\ln(\frac{N}{D}) = \ln(N) - \ln(D)$

2) $\ln(e^{z}) = z$

$$
\ell = -\sum_{j=1}^{C} y_j z_j + \left(\sum_{j=1}^{C} y_j\right) \times\ln\left(\sum_{k=1}^{C} e^{z_k}\right)
$$

Because the labels are one-hot encoded, their sum is 1:

$$
\ell = -\sum_{j=1}^{C} y_j z_j + 1 \times \ln\left(\sum_{k=1}^{C} e^{z_k}\right)
$$

Now differentiate with respect to one logit, $z_r$.

The first term gives $-y_r$. 

For the second term, use the derivative of the natural log together with the chain rule:

$$
\frac{\partial \ell}{\partial z_r}
= -y_r + \frac{e^{z_r}}{\sum_{k=1}^{C} e^{z_k}}
$$

That fraction is the softmax probability $p_r$, so:

$$
\frac{\partial \ell}{\partial z_r} = p_r - y_r
$$

Calculate the average loss for a batch of $B$ observations, where \(i\) is the index on the observations:

$$
\frac{\partial L}{\partial z_{ir}}
= \frac{p_{ir} - y_{ir}}{B}
$$

Now we are back to this statement:

`d_loss__d_z_o = (a_o - y_true) / B`


In [ ]:
# Neural Network with 2 hidden layers
# Uses softmax and cross entropy loss
class MLP_3Layer:
    """
    Variables used
    - w_h1 — Hidden Layer 1 weights
    - b_h1 — Hidden Layer 1 biases
    - z_h1 — Hidden Layer 1 preactivation
    - a_h1 — Hidden Layer 1 activation
    - w_h2 — Hidden Layer 2 weights
    - b_h2 — Hidden Layer 2 biases
    - z_h2 — Hidden Layer 2 preactivation
    - a_h2 — Hidden Layer 2 activation
    - w_o — Output layer weights
    - b_o — Output layer biases
    - z_o — Output layer logits
    - a_o — Output layer probabilities
    """
    def __init__(self, 
                 num_inputs,
                 num_hidden,
                 num_outputs,
                 random_seed=42):
        self.num_inputs = num_inputs
        self.num_hidden = num_hidden
        self.num_outputs = num_outputs
        self.random_seed = random_seed

        # Left to child classes
        self._initialize_weights()

    def _initialize_weights(self):
        np.random.seed(self.random_seed)

        # Initialize the hidden layers weights and biases
        self.w_h1 = np.random.normal(loc=0, scale=0.1, 
                                     size=(self.num_inputs, self.num_hidden))
        self.b_h1 = np.zeros(self.num_hidden)

        self.w_h2 = np.random.normal(loc=0, scale=0.1,
                                     size=(self.num_hidden, self.num_hidden))
        self.b_h2 = np.zeros(self.num_hidden)

        # Initialize output layer weights and biases
        self.w_o = np.random.normal(loc=0, scale=0.1, 
                                    size=(self.num_hidden, self.num_outputs))
        self.b_o = np.zeros(self.num_outputs)

    def forward(self, x):
        """ Feed input data x through each network layer 
            Return a final network output, as well as the intermediate
            values at each layer
        """
        # Apply hidden layer functions
        z_h1 = preactivation(x, self.w_h1, self.b_h1)
        a_h1 = relu(z_h1)

        z_h2 = preactivation(a_h1, self.w_h2, self.b_h2)
        a_h2 = relu(z_h2)

        # Apply output layer function
        z_o = preactivation(a_h2, self.w_o, self.b_o)
        a_o = softmax(z_o)

        predictions = a_o # Output layer probabilities
        
        values_needed_for_backward = (
            a_o,  # Output layer probabilities
            z_o,  # Output layer logits
            a_h2, # Hidden Layer 2 activation
            z_h2, # Hidden Layer 2 preactivation
            a_h1, # Hidden Layer 1 activation
            z_h1  # Hidden Layer 1 preactivation
        )
        return predictions, values_needed_for_backward


    def backward(self, x, y_true, a_o, z_o, a_h2, z_h2, a_h1, z_h1):
        """
        Purpose:
            Performs backpropagation to calculate the gradients of the loss
            with respect to each weight and bias in the neural network.
    
            Backpropagation starts at the output layer and works backward
            through Hidden Layer 2 and then Hidden Layer 1.
    
            The chain rule is used to determine how each weight and bias
            contributed to the final loss.
    
        Input:
            x      - Input observations.
            y_true - Actual target values.
    
            a_o    - Output layer activation (predictions).
            z_o    - Output layer pre-activation values (logits).
    
            a_h2   - Hidden Layer 2 activation values.
            z_h2   - Hidden Layer 2 pre-activation values.
    
            a_h1   - Hidden Layer 1 activation values.
            z_h1   - Hidden Layer 1 pre-activation values.
    
        Output:
            Gradients of the loss with respect to the weights and biases
            for the output layer, Hidden Layer 2, and Hidden Layer 1.
    
        Variable naming:
            Variables beginning with "d_" represent derivatives/gradients.
    
            d_[thing being differentiated]__d_[with respect to]
    
            Example: d_loss__d_a_o
    
            means:    
                derivative of loss with respect to output-layer activation

        Itermediate Arrays and Derivatives:
            using the naming convention described above there are numerous
            intermediate arrays used to store the  necessary derivatives
            
            Their shape and size is controlled by:
            B = number of observations in the current batch
            I = number of input features (784) 
            H = number of neurons in each hidden layer (256)
            O = number of output classes (10)
            
            Output layer
            - d_loss__d_z_o — shape (B, O)
            - d_z_o__d_a_h2 — shape (H, O)
            - d_z_o__d_w_o — shape (B, H)
            - d_loss__d_a_h2 — shape (B, H)
            Hidden Layer 2
            - d_a_h2__d_z_h2 — shape (B, H)
            - d_loss__d_z_h2 — shape (B, H)
            - d_z_h2__d_a_h1 — shape (H, H)
            - d_z_h2__d_w_h2 — shape (B, H)
            - d_loss__d_a_h1 — shape (B, H)
            Hidden Layer 1
            - d_a_h1__d_z_h1 — shape (B, H)
            - d_loss__d_z_h1 — shape (B, H)
            - d_z_h1__d_w_h1 — shape (B, I)
            - d_z_h1__d_x — shape (I, H);

        Notes:
            einsum stands for Einstein summation. 
            I'll be honest it takes awhile to 
            break it down. Would prefer it used a
            more linear algebra equation.
        """

        # B = number of observations in the current batch
        # I = number of input features (784) 
        # H = number of neurons in each hidden layer
        # O = number of output classes (10)
        
        B, I = x.shape # [B, I]
        _, O = y_true.shape # [B, O]
        assert y_true.shape[0] == B

        # This section performs backpropagation from the output layer
        # through both hidden layers.
        # It apply the chain rule to calculate the loss gradients for each layer's
        # weights and biases, and pass gradients backward to the preceding layer.
        #
        # np.einsum performs the required sums of products, including matrix
        # multiplication and summation across observations. Elementwise
        # multiplication applies the ReLU derivatives in the hidden layers.
        #
        # These calculations produce the gradients,  
        # then weight_update() uses them afterward to update 
        # the model's weights and biases.

        #------------------------------------------------------------------------#
        # Output layer starts

        # Softmax combined with multiclass cross-entropy simplifies the
        # gradient of the loss with respect to the output logits.
        # a_o will be an array with shape (B, O)
        # y_true will have shape (B,O)
        d_loss__d_z_o = (a_o - y_true) / B


        d_z_o__d_a_h2, d_z_o__d_w_o, d_z_o__d_b_o = \
            grad_preactivation(a_h2, self.w_o, self.b_o, z_o)

        d_loss__d_w_o = np.einsum(
            'Bo,Bh->ho',
            d_loss__d_z_o,
            d_z_o__d_w_o
        )
        
        d_loss__d_b_o = np.einsum(
            'Bo,->o',
            d_loss__d_z_o,
            d_z_o__d_b_o
        )
        
        d_loss__d_a_h2 = np.einsum(
            'Bo,ho->Bh',
            d_loss__d_z_o,
            d_z_o__d_a_h2
        )
        # Output layer ends
        #------------------------------------------------------------------------#

        #------------------------------------------------------------------------#
        # 2nd hidden layer starts
        d_a_h2__d_z_h2 = grad_relu(z_h2, a_h2) # Derivative
        d_loss__d_z_h2 = d_loss__d_a_h2 * d_a_h2__d_z_h2

        d_z_h2__d_a_h1, d_z_h2__d_w_h2, d_z_h2__d_b_h2 = \
            grad_preactivation(a_h1, self.w_h2, self.b_h2, z_h2)
        d_loss__d_w_h2 = np.einsum('Bo,Bh->ho', d_loss__d_z_h2, d_z_h2__d_w_h2)
        d_loss__d_b_h2 = np.einsum('Bo,->o', d_loss__d_z_h2, d_z_h2__d_b_h2)
        d_loss__d_a_h1 = np.einsum('Bo,ho->Bh', d_loss__d_z_h2, d_z_h2__d_a_h1)        
        # 2nd hidden layer ends
        #------------------------------------------------------------------------#        

        #------------------------------------------------------------------------#
        # 1st hidden layer
        d_a_h1__d_z_h1 = grad_relu(z_h1, a_h1) # Derivative
        d_loss__d_z_h1 = d_loss__d_a_h1 * d_a_h1__d_z_h1

        d_z_h1__d_x, d_z_h1__d_w_h1, d_z_h1__d_b_h1 = \
            grad_preactivation(x, self.w_h1, self.b_h1, z_h1)
        d_loss__d_w_h1 = np.einsum('Bo,Bh->ho', d_loss__d_z_h1, d_z_h1__d_w_h1)
        d_loss__d_b_h1 = np.einsum('Bo,->o', d_loss__d_z_h1, d_z_h1__d_b_h1)
        # 1st hidden layer ends
        #------------------------------------------------------------------------#        

        # B = number of observations in the current batch
        # I = number of input features (784) 
        # H = number of neurons in each hidden layer
        # O = number of output classes (10)
        return (
            # Output layer gradients
            d_loss__d_w_o,   # (H, O)
            d_loss__d_b_o,   # (O,)

            # Hidden Layer 2 gradients
            d_loss__d_w_h2,  # (H, H)
            d_loss__d_b_h2,  # (H,)

            # Hidden Layer 1 gradients            
            d_loss__d_w_h1,  # (I, H)
            d_loss__d_b_h1,  # (H,)
        )
        
    def weight_update(self, grad_w_o, grad_b_o, grad_w_h2, grad_b_h2, 
                      grad_w_h1, grad_b_h1, lr=0.3):
        """ 
        Update weights and biases via gradient descent
        """
        self.w_o -= lr * grad_w_o
        self.b_o -= lr * grad_b_o
        self.w_h2 -= lr * grad_w_h2
        self.b_h2 -= lr * grad_b_h2
        self.w_h1 -= lr * grad_w_h1
        self.b_h1 -= lr * grad_b_h1


**Question 3 and Question 4 are addressed in the class MLP_3Layer above** 

### Question 3 (15 points): 

Modify the "backward" function of your MLP to explicitly compute and
return gradients of the loss with respect to w_output, b_output, w_2, b_2, w_1, b_1. Be sure
to comment your code to keep track of the shapes of intermediate arrays!

**Question 3 is  addressed in the class MLP_3Layer above** 

### Question 4 (5 points): 

Modify the training loop to compute multi-class accuracy and adjust the weights of the two-layer MLP.

**Question 4 is addressed in the class MLP_3Layer above and the training loop below** 

In [ ]:
def batch_loader(X, y, batch_size=32):
    """Shuffle observations and return one batch at a time."""

    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start + batch_size]
        yield X[batch_indices], y[batch_indices]

In [ ]:
def train(model,
          X_train,
          y_train,
          X_val,
          y_val,
          num_epochs=10,
          lr=0.3,
          lr_decay=1.,
          batch_size=128):

    """ Standalone function to execute the mini-batch SGD training loop

    Parameters:
    -----------
    X_train, y_train: Training dataset
    X_val, y_val: Validation dataset
    num_epochs: number of passes through the dataset, use 10 for consistent comparison with LR above
    batch_size: number of samples in each minibatch
    lr: learning rate for gradient updates
    lr_decay: decreases the learning rate, may aid SGD convergence
    """

    # Create lists to store results.
    train_loss, train_acc = [], []
    val_loss, val_acc = [], []

    # Iterate through the epochs.
    for epoch in range(1, num_epochs + 1):

        # Shuffle the training data and process each batch.
        for X_batch, y_batch in batch_loader(
            X_train, y_train, batch_size
        ):

            # Forward pass.
            predictions, layer_values = model.forward(X_batch)

            # Backward pass.
            gradients = model.backward(
                X_batch, y_batch, *layer_values
            )

            # Update the weights and biases.
            model.weight_update(*gradients, lr=lr)

        # Evaluate the training set after this epoch.
        train_predictions, _ = model.forward(X_train)
        # Store training loss
        train_loss.append(categorical_cross_entropy_loss(y_train, train_predictions))
        # Store training accuracy
        train_acc.append(accuracy(y_train, train_predictions))

        # Evaluate the validation set after this epoch.
        val_predictions, _ = model.forward(X_val)
        # Store validation loss
        val_loss.append(categorical_cross_entropy_loss(y_val, val_predictions))
        # Store validation accuracy 
        val_acc.append(accuracy(y_val, val_predictions))

        # Optional: print a brief update every 10 epochs.       
        if (epoch == 1) or (epoch % 10 == 0) or (epoch == num_epochs):
            print(
                f"Epoch {epoch}/{num_epochs} | "
                f"Training loss: {train_loss[-1]:.3f} | "
                f"Training accuracy: {train_acc[-1]:.3f} | "
                f"Validation loss: {val_loss[-1]:.3f} | "
                f"Validation accuracy: {val_acc[-1]:.3f}"
            )

        # Reduce learning rate if lr_decay is below 1.
        lr *= lr_decay

    return train_loss, train_acc, val_loss, val_acc

### Question 5 (10 points): Initialize and train your model. Aim for a minimum training accuracy of 95%.

Starting with 
- Hidden layer width: 256
- Hidden layer activation: ReLU
- Learning rate: 0.001
- Epochs: 50

In [ ]:
model = MLP_3Layer(num_inputs=X_train.shape[1], 
                   num_hidden=512, 
                   num_outputs=y_train.shape[1])

# Record the time immediately before starting the training.
start_time = dt.now()
print(f"Start time: {start_time:%Y-%m-%d %H:%M:%S}")

train_loss, train_acc, val_loss, val_acc = train(
    model, X_train, y_train, X_val, y_val,
    num_epochs=60,
    lr=0.005
)

# Stop the training timer before generating results.
end_time = dt.now()
elapsed_time = end_time - start_time

print(f"End time:     {end_time:%Y-%m-%d %H:%M:%S}")
print(f"Runtime:      {elapsed_time}")

In [ ]:
epochs = range(1, len(train_acc) + 1)

# List positions start at 0, so position 9 represents epoch 10.
# range(start, stop, step)
# start value is 9 which is really 10 because the list starts at 0
# stop value is the number of epochs which is len(train_acc)
# step value is 10 so every 10 epochs we place a marker
marker_positions = list(range(9, len(train_acc), 10))

plt.figure(figsize=(10, 6))

plt.plot(
    epochs, train_acc,
    label="Training Accuracy",
    marker="o",
    markevery=marker_positions
)

plt.plot(
    epochs, val_acc,
    label="Validation Accuracy",
    marker="o",
    markevery=marker_positions
)

# Plot a marker every 10 epochs and show the value
for i in marker_positions:
    # Label training values above their markers 
    plt.annotate(
        f"{train_acc[i]:.3f}",
        (i + 1, train_acc[i]),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=8
    )
    # Label validation values below their markers.
    plt.annotate(
        f"{val_acc[i]:.3f}",
        (i + 1, val_acc[i]),
        textcoords="offset points",
        xytext=(0, -14),
        ha="center",
        fontsize=8
    )

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy using Standard Scaler")
plt.grid()
plt.legend()
plt.tight_layout()
plt.savefig("Assignment1_P1_Accuracy_Standard.png")
plt.show()

In [ ]:
epochs = range(1, len(train_acc) + 1)

# List positions start at 0, so position 9 represents epoch 10.
# range(start, stop, step)
# start value is 9 which is really 10 because the list starts at 0
# stop value is the number of epochs which is len(train_acc)
# step value is 10 so every 10 epochs we place a marker

marker_positions = list(range(9, len(train_acc), 10))

plt.figure(figsize=(10, 6))

plt.plot(
    epochs, train_loss,
    label="Training Loss",
    marker="o",
    markevery=marker_positions
)

plt.plot(
    epochs, val_loss,
    label="Validation Loss",
    marker="o",
    markevery=marker_positions
)

# Plot a marker every 10 epochs and show the value
for i in marker_positions:
    plt.annotate(
        f"{train_loss[i]:.3f}",
        (i + 1, train_loss[i]),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=8
    )

    plt.annotate(
        f"{val_loss[i]:.3f}",
        (i + 1, val_loss[i]),
        textcoords="offset points",
        xytext=(0, -14),
        ha="center",
        fontsize=8
    )

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss using Standard Scaler")
plt.grid()
plt.legend()
plt.tight_layout()
plt.savefig("Assignment1_P1_Loss_Standard.png")
plt.show()

In [ ]:
# Make predictions on the test set here
y_test_pred, _ = model.forward(X_test)
test_loss = categorical_cross_entropy_loss(y_test, y_test_pred)
test_acc = accuracy(y_test, y_test_pred)

print("Using Standard Scaler")
print(f"Test cross-entropy loss = {test_loss:.3f}\tTest accuracy = {test_acc:.3f}")

### Question 5: Comments on the model's performance

Using standard scaling, model trained for 60 epochs with a learning rate of 0.005 and 512 neurons in each hidden layer.

Training accuracy reached 97.0%, exceeding the 95% target, while training loss fell to 0.109. Validation accuracy reached 87.1%, with a validation loss of approximately 0.430.

Most validation improvement occurred during the first 20–30 epochs. 

Afterward, validation loss fluctuated around 0.42–0.43, while validation accuracy improved only slightly. Training performance continued improving, creating a widening gap between training and validation results. Appears to be a sign of overfitting which is typical. 

The model achieved 88.2% test accuracy and a test cross-entropy loss of 0.379. Test performance was slightly better than validation performance, but both were substantially below training accuracy. 

Overall, the model met the training target, while the curves suggest diminishing benefits from additional epochs.

### Question 6 (5 points): Re-initialize the model and train again using a different input feature scaling.

In [ ]:
from sklearn.model_selection import train_test_split

# Always split the original data before scaling 
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Split again from the temp data to create validation and test sets
# You should have 80% for training, then 10% each for validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# Fit the scaler on just the training data. This means
# learning the scaling parameters from X_train.

scaler = MinMaxScaler()
scaler.fit(X_train)

# Apply that same scaler on all three sets
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
# Convert y (the label number 0 thru 9) to a one hot format
# in all the y arrays
y_train = np.eye(10, dtype=np.float32)[y_train]
y_val = np.eye(10, dtype=np.float32)[y_val]
y_test = np.eye(10, dtype=np.float32)[y_test]

Re-initialize the model and train again using a different input feature scaling.

In [ ]:
# Re-initialize the model
model = MLP_3Layer(
    num_inputs=X_train.shape[1],
    num_hidden=512,
    num_outputs=y_train.shape[1]
)

In [ ]:
# Train again using a different input feature scaling.


# Record the time immediately before starting the training.
start_time = dt.now()
print(f"Start time: {start_time:%Y-%m-%d %H:%M:%S}\n")

train_loss, train_acc, val_loss, val_acc = train(
    model, X_train, y_train, X_val, y_val,
    num_epochs=60,
    lr=0.005
)

# Stop the training timer before generating results.
end_time = dt.now()
elapsed_time = end_time - start_time

print(f"Training completed")
print(f"End time:     {end_time:%Y-%m-%d %H:%M:%S}")
print(f"Runtime:      {elapsed_time}")

In [ ]:
# Make predictions on the test set here
y_test_pred, _ = model.forward(X_test)
test_loss = categorical_cross_entropy_loss(y_test, y_test_pred)
test_acc = accuracy(y_test, y_test_pred)

print("Using MinMax Scaler")
print(f"Test cross-entropy loss = {test_loss:.3f}\tTest accuracy = {test_acc:.3f}")

In [ ]:
epochs = range(1, len(train_acc) + 1)

# List positions start at 0, so position 9 represents epoch 10.
marker_positions = list(range(9, len(train_acc), 10))

plt.figure(figsize=(10, 6))

plt.plot(
    epochs, train_acc,
    label="Training Accuracy",
    marker="o",
    markevery=marker_positions
)

plt.plot(
    epochs, val_acc,
    label="Validation Accuracy",
    marker="o",
    markevery=marker_positions
)

# Plot a marker every 10 epochs and show the value
for i in marker_positions:
    plt.annotate(
        f"{train_acc[i]:.3f}",
        (i + 1, train_acc[i]),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=8
    )

    plt.annotate(
        f"{val_acc[i]:.3f}",
        (i + 1, val_acc[i]),
        textcoords="offset points",
        xytext=(0, -14),
        ha="center",
        fontsize=8
    )

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy using MinMax Scaler")
plt.grid()
plt.legend()
plt.tight_layout()
plt.savefig("Assignment1_P1_Accuracy_MinMax.png")
plt.show()

In [ ]:
epochs = range(1, len(train_acc) + 1)

# List positions start at 0, so position 9 represents epoch 10.
marker_positions = list(range(9, len(train_acc), 10))

plt.figure(figsize=(10, 6))

plt.plot(
    epochs, train_loss,
    label="Training Loss",
    marker="o",
    markevery=marker_positions
)

plt.plot(
    epochs, val_loss,
    label="Validation Loss",
    marker="o",
    markevery=marker_positions
)

# Plot a marker every 10 epochs and show the value
for i in marker_positions:
    plt.annotate(
        f"{train_loss[i]:.3f}",
        (i + 1, train_loss[i]),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=8
    )

    plt.annotate(
        f"{val_loss[i]:.3f}",
        (i + 1, val_loss[i]),
        textcoords="offset points",
        xytext=(0, -14),
        ha="center",
        fontsize=8
    )

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss using MinMax Scaler")
plt.grid()
plt.legend()
plt.tight_layout()
plt.savefig("Assignment1_P1_Loss_MinMax.png")
plt.show()

### Question 6: Comments on performance after changing scaling

After reinitializing the model, used min-max scaling, keeping the training configuration at 60 epochs, a learning rate of 0.005, and 512 neurons in each hidden layer.

Test accuracy increased from 88.2% with standard scaling to 89.1% with min-max scaling, an improvement of 0.9 percentage points. 

Test cross-entropy loss decreased from 0.379 to 0.301. 

Final validation accuracy also improved, from 87.1% to 89.0%, while validation loss decreased from 0.430 to 0.324.

However, training accuracy was lower with min-max scaling: 94.0% compared with 97.0%. There was a much smaller gap between training and validation accuracy. No sign of overfitting. The model fit the training data less closely but performed better on unseen data. Always preferred.

In this experiment, min-max scaling appears to have produced a more favorable learning process.

Validation loss continued to decrease gradually, and validation accuracy continued to improve slightly near epoch 60. Additional epochs might have helped, although improvement is not guaranteed and should be monitored using validation performance.